# 02 – Preprocessing and Feature Engineering
**IT3091 Machine Learning · Group 2026-DS-02 · Bank Marketing**

This notebook turns the EDA findings (`docs/eda_insight_log.md`) into justified preprocessing decisions.

**Leakage-prevention principle:** preprocessing is split into two kinds.

| Kind | Examples | Learns from data? | Where it happens |
|---|---|---|---|
| **Row-level cleaning** | drop duplicates, drop `duration`, flags, merges, capping | No (fixed rules) | Here, before the split: `clean_rows()` |
| **Fitted transformations** | one-hot categories, scaling, imbalance handling | Yes | Inside a scikit-learn `Pipeline`, fitted on **training data only** |

All reusable code lives in `src/preprocessing.py`, so every model uses exactly the same preprocessing.

| Section | Content |
|---|---|
| 1 | Setup and loading |
| 2 | Row-level cleaning, step by step |
| 3 | Hold-out train/test split |
| 4 | Feature selection check (training data only) |
| 5 | Multicollinearity fix for the linear model |
| 6 | Build and inspect the two preprocessing pipelines |
| 7 | Class-imbalance strategy |
| 8 | Summary |

## 1. Setup and loading

In [1]:
import sys, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config
from src import preprocessing as prep

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
config.DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(config.RAW_FILE, sep=config.CSV_SEPARATOR)
assert hashlib.sha256(config.RAW_FILE.read_bytes()).hexdigest() == config.RAW_FILE_SHA256, "Wrong dataset file"
print("Raw shape:", raw.shape)

Raw shape: (41188, 21)


## 2. Row-level cleaning, step by step
Each step is shown separately with its effect. At the end we check that the result is identical to
`prep.clean_rows(raw)`, the function the rest of the project uses.

### 2.1 Remove exact duplicates (decision D8, insight E3)
12 rows are exact copies, with no label conflicts. If kept, the same row could appear in both train and test, which slightly inflates test scores.

In [2]:
df = raw.copy()
n_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Rows: {n_before:,} -> {len(df):,} (removed {n_before - len(df)})")
print(f"Yes-rate before: {(raw['y'] == 'yes').mean():.4%}   after: {(df['y'] == 'yes').mean():.4%}")

Rows: 41,188 -> 41,176 (removed 12)
Yes-rate before: 11.2654%   after: 11.2663%


### 2.2 Drop `duration` (decision D3, insight E18)
Call duration is only known **after** the call ends, when the outcome is already known. EDA showed it is the single strongest
predictor (AUC 0.818), so keeping it would give unrealistically good results that cannot be reproduced when ranking clients *before* calling.
(A with/without benchmark is run in the modelling notebook, on training data only.)

In [3]:
df = df.drop(columns=config.LEAKAGE_FEATURES)
print("Dropped:", config.LEAKAGE_FEATURES, "| columns now:", df.shape[1])

Dropped: ['duration'] | columns now: 20


### 2.3 Replace `pdays` with a `contacted_before` flag (decision D10, insight E7)
`pdays = 999` (meaning "not previously contacted") appears in 96% of rows, and 4,110 rows have `pdays = 999` even though `previous > 0`.
Using 999 as a real number of days would be meaningless (and harmful for Logistic Regression). `previous` and `poutcome` are consistent
with each other, so the flag is based on `previous > 0`.

In [4]:
df["contacted_before"] = (df["previous"] > 0).astype(int)

check = pd.crosstab(df["contacted_before"], df["poutcome"])
print("contacted_before vs poutcome (should line up exactly):")
display(check)
rates = df.groupby("contacted_before")["y"].apply(lambda s: round(100 * (s == "yes").mean(), 2))
print("Yes-rate (%) by contacted_before:")
print(rates.to_string())

df = df.drop(columns=["pdays"])

contacted_before vs poutcome (should line up exactly):


poutcome,failure,nonexistent,success
contacted_before,,,
0,0,35551,0
1,4252,0,1373


Yes-rate (%) by contacted_before:
contacted_before
0     8.83
1    26.65


### 2.4 `default`: merge the 3 "yes" rows (decision D13, insight E5)
Only 3 clients have `default = yes`, far too few to learn from, and a one-hot column for them would be almost always 0.
Both "yes" and "unknown" mean *the bank cannot confirm the client is free of default*, so the feature becomes **`no` vs `not_no`**.

In [5]:
print("Before:", df["default"].value_counts().to_dict())
df["default"] = np.where(df["default"] == "no", "no", "not_no")
print("After: ", df["default"].value_counts().to_dict())

Before: {'no': 32577, 'unknown': 8596, 'yes': 3}
After:  {'no': 32577, 'not_no': 8599}


### 2.5 `education`: merge "illiterate" into "basic.4y" (decision D14)
Only 18 rows are "illiterate"; it is the closest level to "basic.4y" (lowest formal education).
Education is **one-hot encoded, not ordinal**: EDA showed the yes-rate is not ordered by education level (basic.9y is lowest, "unknown" is highest),
and "unknown" has no position in an order.

In [6]:
print("illiterate rows:", (df["education"] == "illiterate").sum())
df["education"] = df["education"].replace(prep.EDUCATION_MERGE)
df["education"].value_counts()

illiterate rows: 18


education
university.degree      12164
high.school             9512
basic.9y                6045
professional.course     5240
basic.4y                4194
basic.6y                2291
unknown                 1730
Name: count, dtype: int64

### 2.6 Cap `campaign` at 10 (decision D9, insights E8, E15)
Values above 10 are real but sparse (up to 56) and all have very low yes-rates. Capping keeps their information ("called 10+ times")
while preventing a few extreme values from dominating the scaled Logistic Regression input. Tree models are unaffected either way.

**Why `campaign` is kept although it is only "partly" known before contact (E19):** the model is meant to be re-scored before each call
attempt; at that moment the number of attempts made so far is known to the bank.

In [7]:
n_capped = (df["campaign"] > prep.CAMPAIGN_CAP).sum()
rate_capped = (df.loc[df["campaign"] > prep.CAMPAIGN_CAP, "y"] == "yes").mean()
df["campaign"] = df["campaign"].clip(upper=prep.CAMPAIGN_CAP)
print(f"Rows capped: {n_capped} ({n_capped / len(df):.2%}); their yes-rate: {rate_capped:.2%}")
print("New max campaign:", df["campaign"].max())

Rows capped: 869 (2.11%); their yes-rate: 3.11%
New max campaign: 10


### 2.7 Encode the target and verify against `clean_rows()`
`y`: yes → 1, no → 0. Then the step-by-step result must equal the shared function's output exactly.

In [8]:
df[config.TARGET] = (df[config.TARGET] == config.POSITIVE_CLASS).astype(int)

clean = prep.clean_rows(raw)
pd.testing.assert_frame_equal(df, clean)
print("Step-by-step result == prep.clean_rows(raw)  ✔")

clean.to_csv(config.CLEAN_FILE, index=False)
print("Saved:", config.CLEAN_FILE.relative_to(config.PROJECT_ROOT), clean.shape)
clean.head()

Step-by-step result == prep.clean_rows(raw)  ✔
Saved: data\processed\bank_clean.csv (41176, 20)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,contacted_before
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
1,57,services,married,high.school,not_no,no,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
2,37,services,married,high.school,no,yes,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
4,56,services,married,high.school,no,no,yes,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0


## 3. Hold-out train/test split
The test set is created **now**, before any further decision that looks at the data (feature selection, VIF), so those decisions
are made on training data only. The test set is then **not touched again until final evaluation**.

- **80 / 20 split, stratified on `y`**, so both parts keep the same yes-rate.
- `random_state = 42` for reproducibility.
- The effect of time drift (EDA insight E14) is examined separately with a time-based check in the validation step (decision D12).

In [9]:
X = clean.drop(columns=config.TARGET)
y = clean[config.TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=config.TEST_SIZE, stratify=y, random_state=config.RANDOM_STATE)

X_train.assign(**{config.TARGET: y_train}).to_csv(config.TRAIN_FILE, index=False)
X_test.assign(**{config.TARGET: y_test}).to_csv(config.TEST_FILE, index=False)

pd.DataFrame({
    "rows": [len(y_train), len(y_test)],
    "yes": [y_train.sum(), y_test.sum()],
    "yes_rate_%": [round(100 * y_train.mean(), 2), round(100 * y_test.mean(), 2)],
}, index=["train", "test"])

,rows,yes,yes_rate_%
train,32940,3711,11.27
test,8236,928,11.27


## 4. Feature selection check (training data only)
EDA suggested `housing` and `loan` have no association with the target, and `day_of_week` only a very weak one.
Two pieces of evidence are checked on the **training set**:
1. statistical association (chi-square p-value, Cramér's V);
2. whether removing them changes cross-validated performance of two quick, **untuned** models
   (Logistic Regression and Random Forest). These models only compare feature sets; real modelling happens later.

In [10]:
def cramers_v(x, target):
    table = pd.crosstab(x, target)
    chi2, p, _, _ = stats.chi2_contingency(table)
    v = np.sqrt(chi2 / (table.to_numpy().sum() * (min(table.shape) - 1)))
    return p, v

rows = []
for c in ["housing", "loan", "day_of_week", "poutcome"]:
    p, v = cramers_v(X_train[c], y_train)
    rows.append({"feature": c, "p_value": round(p, 4), "cramers_v": round(v, 3)})
print("Association with the target on the TRAINING set (poutcome shown for comparison):")
pd.DataFrame(rows).set_index("feature")

Association with the target on the TRAINING set (poutcome shown for comparison):


,p_value,cramers_v
feature,,
housing,0.2599,0.009
loan,0.9218,0.002
day_of_week,0.0307,0.018
poutcome,0.0000,0.325


In [11]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=config.RANDOM_STATE)
feature_sets = {
    "all features": [],
    "drop housing + loan": ["housing", "loan"],
    "drop housing + loan + day_of_week": ["housing", "loan", "day_of_week"],
}
models = {
    "LogReg": ("linear", LogisticRegression(class_weight="balanced", max_iter=2000)),
    "RandomForest": ("tree", RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                                                     class_weight="balanced_subsample",
                                                     n_jobs=-1, random_state=config.RANDOM_STATE)),
}

rows = []
for model_name, (kind, model) in models.items():
    for set_name, drop in feature_sets.items():
        pipe = Pipeline([("prep", prep.build_preprocessor(kind, drop=drop)), ("model", model)])
        scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=["average_precision", "roc_auc"])
        rows.append({"model": model_name, "feature set": set_name,
                     "PR-AUC mean": scores["test_average_precision"].mean(),
                     "PR-AUC std": scores["test_average_precision"].std(),
                     "ROC-AUC mean": scores["test_roc_auc"].mean(),
                     "ROC-AUC std": scores["test_roc_auc"].std()})
fs_results = pd.DataFrame(rows).round(4)
fs_results

,model,feature set,PR-AUC mean,PR-AUC std,ROC-AUC mean,ROC-AUC std
0,LogReg,all features,0.4365,0.0126,0.7876,0.0072
1,LogReg,drop housing + loan,0.4368,0.0131,0.7878,0.0071
2,LogReg,drop housing + loan + day_of_week,0.4351,0.0127,0.7881,0.0081
3,RandomForest,all features,0.4641,0.0159,0.7941,0.0073
4,RandomForest,drop housing + loan,0.4646,0.0180,0.7958,0.0093
5,RandomForest,drop housing + loan + day_of_week,0.4593,0.0159,0.7935,0.0078


**How to read this:** if dropping a feature set changes the mean score by much less than the fold-to-fold standard deviation,
the dropped features add no measurable value. Removing them then gives a simpler, more interpretable model at no cost.

## 5. Multicollinearity fix for the linear model (decision D11, insight E13)
EDA found `euribor3m`, `emp.var.rate` and `nr.employed` are almost the same signal (Spearman ρ ≈ 0.94, VIF 32–64).
For Logistic Regression this makes coefficients unstable and uninterpretable, so the **linear pipeline keeps only one of the three**.
Tree models keep all five indicators (they are not affected by collinearity).

Which one to keep: their single-feature predictive strength is checked on the training set.

In [12]:
trio = ["euribor3m", "emp.var.rate", "nr.employed"]
def auc_strength(feature):
    auc = roc_auc_score(y_train, X_train[feature])
    return round(max(auc, 1 - auc), 3)   # direction-free: 0.5 = useless, 1 = perfect

auc_trio = {c: auc_strength(c) for c in trio}
print("Single-feature AUC on training data:", auc_trio)

def vif_table(data):
    rows = []
    for c in data.columns:
        others = data.drop(columns=c)
        r2 = LinearRegression().fit(others, data[c]).score(others, data[c])
        rows.append({"feature": c, "VIF": round(1 / (1 - r2), 1) if r2 < 1 else np.inf})
    return pd.DataFrame(rows).set_index("feature")

_, nums_tree = prep.feature_lists("tree")
_, nums_linear = prep.feature_lists("linear")
vif_compare = vif_table(X_train[nums_tree]).join(
    vif_table(X_train[nums_linear]).rename(columns={"VIF": "VIF after (linear set)"}), how="left")
vif_compare.rename(columns={"VIF": "VIF before (all numerics)"}).fillna("removed")

Single-feature AUC on training data: {'euribor3m': 0.74, 'emp.var.rate': 0.713, 'nr.employed': 0.745}


,VIF before (all numerics),VIF after (linear set)
feature,,
age,1.0,removed
campaign,1.0,1.0
previous,4.8,4.7
contacted_before,4.7,4.7
emp.var.rate,33.5,removed
cons.price.idx,6.3,2.1
cons.conf.idx,2.6,1.2
euribor3m,64.6,2.7
nr.employed,31.7,removed


`euribor3m` is kept for the linear model: its predictive strength is practically the same as `nr.employed`, and it is a
**daily-published, directly interpretable interest rate** that a marketing team can actually monitor when planning a campaign.
Note: `previous` and `contacted_before` are related by construction (the flag is `previous > 0`); a moderate VIF between them is expected and acceptable.

## 6. Build and inspect the two preprocessing pipelines
| | Linear pipeline (Logistic Regression) | Tree pipeline (Decision Tree, Random Forest, XGBoost) |
|---|---|---|
| Categoricals | One-hot | One-hot |
| Age | Age groups (one-hot), captures the U-shape | Raw age (trees find the U-shape themselves) |
| Other numerics | Standard-scaled | Unscaled (trees split on thresholds, so scale does not matter) |
| Economic indicators | `euribor3m`, `cons.price.idx`, `cons.conf.idx` | All five |
| Dropped | `housing`, `loan` | `housing`, `loan` |

Below, each pipeline is fitted on the **training data only** to inspect its output. In modelling, the preprocessor is always placed
inside a `Pipeline` with the model, so during cross-validation it is re-fitted on each training fold.

In [13]:
for kind in ["linear", "tree"]:
    pre = prep.build_preprocessor(kind)
    Xt_train = pre.fit_transform(X_train)          # fit on TRAIN only
    Xt_test = pre.transform(X_test)                # test is only transformed
    names = pre.get_feature_names_out()
    cats, nums = prep.feature_lists(kind)
    print(f"=== {kind} pipeline ===")
    print(f"Input: {len(cats)} categorical + {len(nums)} numeric" + (" + age (binned)" if kind == "linear" else ""))
    print(f"Output columns: {len(names)} | train {Xt_train.shape} | test {Xt_test.shape}")
    print(f"Any missing values after transform: {np.isnan(Xt_train).any() or np.isnan(Xt_test).any()}")
    print("Columns:", list(names))
    print()

=== linear pipeline ===
Input: 8 categorical + 6 numeric + age (binned)
Output columns: 58 | train (32940, 58) | test (8236, 58)
Any missing values after transform: False
Columns: ['job_admin.', 'job_blue-collar', 'job_entrepreneur', 'job_housemaid', 'job_management', 'job_retired', 'job_self-employed', 'job_services', 'job_student', 'job_technician', 'job_unemployed', 'job_unknown', 'marital_divorced', 'marital_married', 'marital_single', 'marital_unknown', 'education_basic.4y', 'education_basic.6y', 'education_basic.9y', 'education_high.school', 'education_professional.course', 'education_university.degree', 'education_unknown', 'default_no', 'default_not_no', 'contact_cellular', 'contact_telephone', 'month_apr', 'month_aug', 'month_dec', 'month_jul', 'month_jun', 'month_mar', 'month_may', 'month_nov', 'month_oct', 'month_sep', 'day_of_week_fri', 'day_of_week_mon', 'day_of_week_thu', 'day_of_week_tue', 'day_of_week_wed', 'poutcome_failure', 'poutcome_nonexistent', 'poutcome_success',

In [14]:
# Scaling check for the linear pipeline: scaled numerics should have mean ~0 and std ~1 on TRAIN
pre_lin = prep.build_preprocessor("linear").fit(X_train)
Xt = pd.DataFrame(pre_lin.transform(X_train), columns=pre_lin.get_feature_names_out())
_, nums_linear = prep.feature_lists("linear")
Xt[nums_linear].describe().loc[["mean", "std"]].round(3)

,campaign,previous,contacted_before,euribor3m,cons.price.idx,cons.conf.idx
mean,0.0,0.0,-0.0,-0.0,0.0,0.0
std,1.0,1.0,1.0,1.0,1.0,1.0


## 7. Class-imbalance strategy (decision D16, insight E1)
The training data is about 7.9 : 1 (no : yes). Options:

| Option | How | Pros | Cons |
|---|---|---|---|
| Do nothing | – | simple | models favour "no", recall for "yes" is low |
| Random undersampling | drop "no" rows | fast | throws away ~75% of the data |
| SMOTE oversampling | create synthetic "yes" rows | balances classes | synthetic clients; must be applied **inside** CV folds only, or it leaks |
| **Class weights** | errors on "yes" cost ~7.9× more | no data changed, no leakage risk, supported by all 4 models | probability outputs are shifted (fine for ranking) |
| Threshold tuning | choose the cut-off for "yes" | directly matches business cost | done after modelling |

**Chosen:** class weights as the main approach (`class_weight="balanced"` for sklearn models, `scale_pos_weight` for XGBoost),
SMOTE (inside an `imblearn` pipeline) as a compared alternative in modelling, and threshold tuning in evaluation.
**Nothing is resampled here**: resampling before the split or outside CV folds would leak information.

In [15]:
classes = np.array([0, 1])
weights = compute_class_weight("balanced", classes=classes, y=y_train)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("Training counts:", y_train.value_counts().to_dict())
print(f"class_weight='balanced' -> no: {weights[0]:.3f}, yes: {weights[1]:.3f}")
print(f"XGBoost scale_pos_weight = {scale_pos_weight:.2f}")

Training counts: {0: 29229, 1: 3711}
class_weight='balanced' -> no: 0.563, yes: 4.438
XGBoost scale_pos_weight = 7.88


## 8. Summary
Full details: `docs/preprocessing_findings.txt` and `docs/preprocessing_log.md`.

**Row-level cleaning** (`prep.clean_rows`, verified identical to the step-by-step version)
- 12 duplicates removed: 41,188 → **41,176 rows**; yes-rate unchanged (11.27%).
- `duration` dropped (post-call leakage).
- `pdays` replaced by `contacted_before`. The flag lines up **exactly** with `poutcome` (0 ↔ nonexistent; 1 ↔ failure/success), confirming it is consistent. Yes-rate 8.83% vs 26.65%.
- `default` → `no` (32,577) vs `not_no` (8,599); `education` "illiterate" (18) merged into "basic.4y".
- `campaign` capped at 10: 869 rows (2.11%) affected, with a yes-rate of only 3.11%.
- Saved `bank_clean.csv` (41,176 × 20).

**Hold-out split:** stratified 80/20, seed 42. Train 32,940 / test 8,236; both exactly **11.27% yes**. Test set locked until final evaluation.

**Feature selection (training data only)**
- `housing` (p = 0.26, V = 0.009) and `loan` (p = 0.92, V = 0.002): no significant association.
- Dropping them changes CV PR-AUC by only +0.0003 (LR) / +0.0005 (RF), far below the fold std (≈ 0.013–0.018), so they are **dropped** (simpler model, no loss).
- `day_of_week` is weak but significant (p = 0.03), and dropping it lowered PR-AUC in **both** models (−0.0017 LR, −0.0048 RF), so it is **kept**.
- Untuned models already reach PR-AUC ≈ 0.44–0.46 (vs 0.11 for random guessing) and ROC-AUC ≈ 0.79, an early indication that the features carry real signal.

**Multicollinearity (linear model only):** keeping `euribor3m` and dropping `emp.var.rate`/`nr.employed` reduces the maximum VIF from **64.6 to 4.7**; all linear inputs are now below 5. `euribor3m` (AUC 0.740) is practically as strong as `nr.employed` (0.745) and is easier to interpret.

**Pipelines:** linear = 58 columns (one-hot + age groups + scaled numerics, mean 0 / std 1); tree = 54 columns (one-hot + raw numerics, all economic indicators). No missing values in either.

**Imbalance:** train 29,229 no / 3,711 yes. Class weights no = 0.563, yes = 4.438; XGBoost `scale_pos_weight` = 7.88. SMOTE compared in modelling; threshold tuned in evaluation.